In [ ]:
from constants import *
from astropy import units as u

from phoenix_grid_creator.spectral_grid import spectral_grid
from spectrum_component_analyser.phoenix_spectrum import phoenix_spectrum

fits_file_paths = list(Path(package_path / "raw_phoenix_spectra").rglob("*.fits"))

In [ ]:
# fits_file_paths =fits_file_paths[:1]
import numpy as np

resolution : u.Quantity[u.K] = .01*u.um
observational_wavelengths : np.ndarray = np.arange(0.8, 5.3, resolution.value) * u.um
spec_grid : spectral_grid = spectral_grid.from_local_raw(
    fits_file_paths,
    parallelise=False,
    observational_wavelengths=observational_wavelengths)



In [ ]:
# units are getting lost somewhere, idk why
spec_grid.Wavelengths = spec_grid.Wavelengths.to(u.um)
spec_grid.Fluxes *= u.Jy

print(fits_file_paths)
print(spec_grid.T_effs[0])
print(spec_grid.FeHs[0])
print(spec_grid.Log_gs[0])

spec_grid.get_spectrum(spec_grid.T_effs[0], spec_grid.FeHs[0], spec_grid.Log_gs[0]).plot()

In [ ]:
def remove_nan_wavelengths(spec_grid : spectral_grid) -> None:
    """
    Identifies indices where NaNs exist in the flux array across any 
    parameter combination and removes them from both Wavelengths and Fluxes.
    """
    # 1. Check for NaNs along the parameter axes (0, 1, 2), leaving wavelength axis (3)
    # This creates a boolean array of length = number of wavelengths
    nan_mask = np.any(np.isnan(spec_grid.Fluxes), axis=(0, 1, 2))
    
    # 2. Invert mask to keep only the 'good' wavelengths (where no NaN exists)
    keep_indices = ~nan_mask
    
    # 3. Update the class attributes
    # Update Wavelengths (ensure it remains an astropy Quantity)
    spec_grid.Wavelengths = spec_grid.Wavelengths[keep_indices]
    
    # Update Fluxes (slicing the 4th dimension)
    spec_grid.Fluxes = spec_grid.Fluxes[:, :, :, keep_indices]
    
    # 4. Refresh the lookup table since the data has changed
    spec_grid.LookupTable = spec_grid.to_lookup_table()
    
    print(f"[PHOENIX GRID] : Removed {np.sum(nan_mask)} wavelength points containing NaNs.")

remove_nan_wavelengths(spec_grid)

In [ ]:
%matplotlib inline
# take 2 random spectra
# combine them with random weights
# see if you can reach a global minimum at a given resolution
# repeat for many resolutions


from matplotlib import pyplot as plt

import rich
from spectrum_component_analyser.mcmc.simulated_spectra import get_random_simulated_spectrum, get_simulated_spectra
from spectrum_component_analyser.spectral_component import spectral_component
from spectrum_component_analyser.spectrum import spectrum

# number_of_components : int = 1
# true_components, component_spectra, combined_spectrum = get_random_simulated_spectrum(number_of_components, spec_grid, FeH=0.0 * u.dex, log_g=4.5 * u.dex)
 
true_feh = 0.0 * u.dex
true_logg = 4.5 *  u.dex

true_components : list[spectral_component] = [
    spectral_component(3800 * u.K, true_feh, true_logg, 0.85),
    spectral_component(3300 * u.K, true_feh, true_logg, .15),
]
number_of_components = len(true_components)

true_components, component_spectra, combined_spectrum = get_simulated_spectra(spec_grid, true_components)

true_table = spectral_component.return_default_table()
for c in true_components:
    c.pretty_print(true_table)
rich.print(true_table)

In [ ]:
import numpy as np
import scipy as sp
from spectrum_component_analyser.interpolated_spectrum import get_interpolated_phoenix_spectrum
from spectrum_component_analyser.chi_squared_minimisation import ChiHelper, get_chi_r

# both actual global constants
NUMBER_OF_PARAMETERS : int = 4 # weight, t, f, l



r = get_chi_r(number_of_components, NUMBER_OF_PARAMETERS, combined_spectrum, spec_grid)

In [ ]:
%matplotlib inline

# print("simulated / true")
# rich.print(true_table)

# chi.plot(r)

In [ ]:
from spectrum_component_analyser.mcmc.constrained_helper import ConstrainedMCMCHelper

parameter_bounds = [
        (.0, 2.),
        (np.min(spec_grid.T_effs.value), np.max(spec_grid.T_effs.value)),
        (np.min(spec_grid.FeHs.value), np.max(spec_grid.FeHs.value)),
        (np.min(spec_grid.Log_gs.value), np.max(spec_grid.Log_gs.value)),
    ]

mcmc : ConstrainedMCMCHelper = ConstrainedMCMCHelper(
    parameter_bounds=parameter_bounds,
    number_of_components=number_of_components,
    observed_spectrum=combined_spectrum,
    spec_grid=spec_grid,
    n_steps=20000,
    n_walkers=int(2.5 * number_of_components * NUMBER_OF_PARAMETERS)
)

sampler, samples = mcmc.run(r)

print(int(2.5 * number_of_components * NUMBER_OF_PARAMETERS))

In [ ]:
import corner
import emcee

%matplotlib inline

def plot_corner(mcmc : ConstrainedMCMCHelper, sampler : emcee.EnsembleSampler, samples : np.ndarray, true_components : list[spectral_component]):
    labels = []
    for i in range(mcmc.number_of_components):
        labels += [f"$f_{i+1}$", f"$T_{i+1}$ [K]"]
    labels += ["[Fe/H] [dex]", r"$\log g$ [dex]"]

    main_fill_color = "#FF8B00"
    quantile_line_color = "#AC8DD9"

    # fig, axes = plt.subplots(samples.shape[1], samples.shape[1], figsize=(16, 16))
    
    fontsize = 12
    plt.rc('xtick', labelsize=fontsize - 2)
    plt.rc('ytick', labelsize=fontsize - 2)

    true_params = []
    for c in true_components:
        true_params += [c.Weight, c.T_eff.value]
    true_params += [true_feh.value, true_logg.value]
    print(true_feh)
    print(true_logg)
    
    true_params = np.array(true_params, dtype=float)

    truth_color = "black"

    fig = corner.corner(
        samples,
        truths=true_params,
        truth_color=truth_color,
        # fig=fig,
        labels=labels, 
        quantiles=[0.16, 0.5, 0.84],
        show_titles=True,
        # --- Neatness Settings ---
        color=main_fill_color,
        plot_datapoints=False,         # Removes the scatter points
        plot_density=False,            # Removes the low-res "square" histogram
        fill_contours=True,         # Keeps it looking minimal
        levels=(1 - np.exp(-0.5 * np.arange(1, 4)**2)), # Standard 1, 2, 3 sigma
        contour_kwargs={"colors": [main_fill_color], "linewidths": 1.0},
        hist_kwargs={
            "color": "black",             # This sets the 'edgecolor' when histtype='step'
            "fill": True,                 # Enables the fill
            "facecolor": main_fill_color, # The color inside the histograms
            "edgecolor": "black",         # The outline color
            "linewidth": 1.5,
            "alpha": 0.5                  # Slight transparency often looks cleaner
        },
        title_kwargs={"fontsize": fontsize},
        label_kwargs={"fontsize": fontsize},     
        )
    
    axes = np.array(fig.axes).reshape((samples.shape[1], samples.shape[1]))

    for i in range(samples.shape[1]):
        ax = axes[i, i] # Target the 1D histograms on the diagonal

        current_title = ax.get_title()
        if "T_" in labels[i]:
            value = current_title.split("=")[1]
            ax.set_title(r"$T_\text{eff}$ =" + f"{value} K", fontsize=fontsize)
        elif "Fe/H" in labels[i]:
            value = current_title.split("=")[1]
            ax.set_title(r"[Fe/H] =" + f"{value} dex", fontsize=fontsize)
        elif "log" in labels[i]:
            value = current_title.split("=")[1]
            ax.set_title(r"$\log g$ =" + f"{value} dex", fontsize=fontsize)
        
        # Corner usually plots quantile lines as 'Line2D' objects
        idx = 0
        for line in ax.get_lines():
            if line.get_color() == truth_color:
                continue
            if (idx == 1):
                line.set_linestyle("-")
            # only count the lines that aren't truth-lines
            idx += 1

            # line.set_color("#705CC8B0")
            # line.set_linestyle("--") # Optional: make them dashed
            # line.set_linewidth(2)     # Optional: make them thicker
    
    for ax in fig.axes:
        for line in ax.get_lines():
            if line.get_color() == truth_color:
                line.set_linewidth(1)
                line.set_markersize(5)
                line.set_zorder(10)

    plt.rcParams['xtick.direction'] = 'in'
    plt.rcParams['ytick.direction'] = 'in'
    # Optional: makes ticks appear on all sides
    plt.rcParams['xtick.top'] = True
    plt.rcParams['ytick.right'] = True

    # fig.suptitle("Posterior distributions for a simulated M dwarf", fontsize=20)
    fig.subplots_adjust(top=0.93)
    plt.show()

    # caterpillar
    fig, axes = plt.subplots(mcmc.ndim, figsize=(10, 7), sharex=True)
    chain = sampler.get_chain()
    for i in range(mcmc.ndim):
        ax = axes[i]
        ax.plot(chain[:, :, i], "k", alpha=0.3)
        ax.set_xlim(0, len(chain))
        ax.set_ylabel(labels[i])
        ax.yaxis.set_label_coords(-0.1, 0.5)

    axes[-1].set_xlabel("Step number")
    # plt.tight_layout()
    plt.show()

plot_corner(mcmc, sampler, samples, true_components)

print("\n[ORIGINAL PARAMETERS]")
rich.print(true_table)

mcmc.print_parameters(samples=samples)

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'svg'

plt.clf()

from matplotlib.ticker import AutoMinorLocator, FixedLocator, MaxNLocator, ScalarFormatter

def plot_highres_spectrum(mcmc : ConstrainedMCMCHelper, samples : np.ndarray) -> None:
    best_params = np.median(samples, axis=0)
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 10), sharex=True, gridspec_kw={'height_ratios': [3, 1, 1]}, dpi=150)
    

    ax1.plot(mcmc.ObservedSpectrum.Wavelengths, mcmc.ObservedSpectrum.Fluxes, label="Original simulated spectrum (ground truth)", color="black", linewidth=.2, alpha=1)

    total_flux = np.zeros(len(mcmc.ObservedSpectrum.Wavelengths)) * u.Jy
    shared_f = best_params[-2] * u.dex
    shared_l = best_params[-1] * u.dex

    
    colors = ["royalblue", "forestgreen"]
    for i in range(mcmc.number_of_components):
        w = best_params[i*2]
        t = best_params[i*2 + 1] * u.K
        
        comp = get_interpolated_phoenix_spectrum(t, shared_f, shared_l, star_name="modelled spectrum", spec_grid=mcmc.spec_grid)
        weighted_flux = comp.Fluxes * w
        total_flux += weighted_flux

        ax1.plot(mcmc.ObservedSpectrum.Wavelengths,
                 weighted_flux.to(u.Jy), 
                 label=f"Best fit - component {i+1} | $f$ = {w.round(2)}, $T$ = {t.value:.0f} K, $\log g$ = {shared_l.round(2)}, [Fe/H] = {shared_f.round(2)}",
                 alpha=0.8,
                 linestyle='--',
                 linewidth=.5,
                 color=colors[i])

    ax1.plot(mcmc.ObservedSpectrum.Wavelengths, total_flux, label="Best fit - total", color="orange", linewidth=.5, alpha=0.8)
    res = (total_flux.value - mcmc.ObservedSpectrum.Fluxes.value) / mcmc.ObservedSpectrum.Fluxes.value
    ax2.plot(mcmc.ObservedSpectrum.Wavelengths, res, color="#350AC4", linewidth=.5, alpha=0.8)
    ax3.plot(mcmc.ObservedSpectrum.Wavelengths, np.log(np.abs(res)), color="#350AC4", linewidth=.5, alpha=.8)

    # FORMATTING
    ax1.xaxis.set_minor_locator(AutoMinorLocator(5))
    ax1.yaxis.set_minor_locator(AutoMinorLocator(5))
    
    ax1.xaxis.set_minor_formatter(ScalarFormatter())
    ax1.yaxis.set_minor_formatter(ScalarFormatter())

    ax1.tick_params(axis='both', which='major', labelsize=12)
    ax1.tick_params(axis='both', which='minor', labelsize=9, pad=3)
    ax1.tick_params(which='minor', length=2, width=0.5)

    ax1.grid(which='major', color='#CCCCCC', linestyle='-', linewidth=2, alpha=0.7)
    ax1.grid(which='minor', color='#DDDDDD', linestyle=':', linewidth=2, alpha=1)
    leg = ax1.legend(fontsize=14)
    for line in leg.get_lines():
        line.set_alpha(1.0)
        line.set_linewidth(2.0)
    ax1.set_ylabel("Normalised Intensity [arbitrary units]", size=14, labelpad=20)
    ax1.set_ylim(0)

    major_ticks = [0, 1, 2, 3]
    major_locator = FixedLocator(major_ticks)

    # Apply to the x-axis
    ax2.yaxis.set_major_locator(major_locator)
    # second axis
    ax2.xaxis.set_minor_locator(AutoMinorLocator())
    ax2.yaxis.set_minor_locator(AutoMinorLocator(2))

    ax3.xaxis.set_minor_locator(AutoMinorLocator())
    ax3.yaxis.set_minor_locator(AutoMinorLocator(2))

    ax2.set_ylabel(r"Residual = $\frac{\mathrm{Best\ fit}-\mathrm{Original}}{\mathrm{Original}}$", size=14, labelpad=20)
    ax3.set_ylabel(r"$\log($Residual$)$", size=14, labelpad=20)
    for ax in [ax2, ax3]:
        
        ax.xaxis.set_minor_formatter(ScalarFormatter())
        ax.yaxis.set_minor_formatter(ScalarFormatter())

        ax.tick_params(axis='both', which='major', labelsize=12)
        ax.tick_params(axis='both', which='minor', labelsize=9, pad=3)
        ax.tick_params(which='minor', length=2, width=0.5)

        ax.grid(which='major', color='#CCCCCC', linestyle='-', linewidth=2, alpha=0.7)
        ax.grid(which='minor', color='#DDDDDD', linestyle=':', linewidth=2, alpha=1)
        
        # FINISHING UP
    ax3.set_xlabel("Wavelength [μm]", size=14, labelpad=20)

    ax2.set_ylim(-1, 3)
    plt.tight_layout()
    plt.show()

plot_highres_spectrum(mcmc, samples)

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'svg'

plt.clf()

from matplotlib.ticker import AutoMinorLocator, FixedLocator, MaxNLocator, ScalarFormatter

def plot_lowres_spectrum(mcmc : ConstrainedMCMCHelper, samples : np.ndarray) -> None:
    best_params = np.median(samples, axis=0)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True, gridspec_kw={'height_ratios': [3, 1]}, dpi=150)
    

    ax1.plot(mcmc.ObservedSpectrum.Wavelengths, mcmc.ObservedSpectrum.Fluxes, label="Original simulated spectrum (ground truth)", color="black", alpha=0.8)

    total_flux = np.zeros(len(mcmc.ObservedSpectrum.Wavelengths)) * u.Jy
    shared_f = best_params[-2] * u.dex
    shared_l = best_params[-1] * u.dex

    
    colors = ["royalblue", "forestgreen"]
    for i in range(mcmc.number_of_components):
        w = best_params[i*2]
        t = best_params[i*2 + 1] * u.K
        
        comp = get_interpolated_phoenix_spectrum(t, shared_f, shared_l, star_name="modelled spectrum", spec_grid=mcmc.spec_grid)
        weighted_flux = comp.Fluxes * w
        total_flux += weighted_flux

        ax1.plot(mcmc.ObservedSpectrum.Wavelengths,
                 weighted_flux.to(u.Jy), 
                 label=f"Best fit - component {i+1} | $f$ = {w.round(2)}, $T$ = {t.value:.0f} K, $\log g$ = {shared_l.round(2)}, [Fe/H] = {shared_f.round(2)}",
                #  alpha=0.8,
                 linestyle='--',
                #  linewidth=.5,
                 color=colors[i])

    ax1.plot(mcmc.ObservedSpectrum.Wavelengths, total_flux, label="Best fit - total", color="orange",alpha=0.8, linewidth=2)
    res = (total_flux.value - mcmc.ObservedSpectrum.Fluxes.value) / mcmc.ObservedSpectrum.Fluxes.value
    ax2.plot(mcmc.ObservedSpectrum.Wavelengths, res, color="blue")

    # FORMATTING
    ax1.xaxis.set_minor_locator(AutoMinorLocator(5))
    ax1.yaxis.set_minor_locator(AutoMinorLocator(5))
    
    ax1.xaxis.set_minor_formatter(ScalarFormatter())
    ax1.yaxis.set_minor_formatter(ScalarFormatter())

    ax1.tick_params(axis='both', which='major', labelsize=12)
    ax1.tick_params(axis='both', which='minor', labelsize=9, pad=3)
    ax1.tick_params(which='minor', length=2, width=0.5)

    ax1.grid(which='major', color='#CCCCCC', linestyle='-', linewidth=2, alpha=0.7)
    ax1.grid(which='minor', color='#DDDDDD', linestyle=':', linewidth=2, alpha=1)
    leg = ax1.legend(fontsize=14)
    ax1.set_ylabel("Normalised Intensity [arbitrary units]", size=14, labelpad=20)
    ax1.set_ylim(0)

    major_ticks = [0, 1, 2, 3]
    major_locator = FixedLocator(major_ticks)

    # Apply to the x-axis
    ax2.yaxis.set_major_locator(major_locator)
    # second axis
    ax2.xaxis.set_minor_locator(AutoMinorLocator())
    ax2.yaxis.set_minor_locator(AutoMinorLocator(2))

    ax2.set_ylabel(r"Residual = $\frac{\mathrm{Best\ fit}-\mathrm{Original}}{\mathrm{Original}}$", size=14, labelpad=20)
    for ax in [ax2]:
        
        ax.xaxis.set_minor_formatter(ScalarFormatter())
        ax.yaxis.set_minor_formatter(ScalarFormatter())

        ax.tick_params(axis='both', which='major', labelsize=12)
        ax.tick_params(axis='both', which='minor', labelsize=9, pad=3)
        ax.tick_params(which='minor', length=2, width=0.5)

        ax.grid(which='major', color='#CCCCCC', linestyle='-', linewidth=2, alpha=0.7)
        ax.grid(which='minor', color='#DDDDDD', linestyle=':', linewidth=2, alpha=1)
        
        # FINISHING UP
    ax2.set_xlabel("Wavelength [μm]", size=14, labelpad=20)

    ax2.set_ylim(-1, 3)
    plt.tight_layout()
    plt.show()

plot_lowres_spectrum(mcmc, samples)

In [ ]:
def get_viable_samples_strict(sampler, percentage=0.01):
    """
    Retrieves samples within a specific percentage of the peak likelihood.
    A 1% drop in likelihood (percentage=0.01) is a very strict constraint.
    """
    log_prob = sampler.get_log_prob(flat=True)
    flat_samples = sampler.get_chain(flat=True)
    
    # Peak log-probability
    max_lp = np.max(log_prob)
    
    # Likelihood L = exp(log_prob)
    # We want L > (1 - percentage) * L_max
    # ln(L) > ln(1 - percentage) + ln(L_max)
    # Delta log_prob < -ln(1 - percentage)
    log_prob_threshold = -np.log(1.0 - percentage)
    
    viable_mask = (max_lp - log_prob) < log_prob_threshold
    
    return flat_samples[viable_mask], log_prob_threshold

# Get samples within 10% of the maximum likelihood
viable_samples, threshold = get_viable_samples_strict(sampler, percentage=.1)

print(f"Strict Maximum Likelihood Threshold (Delta Log-Prob): {threshold:.4f}")
print(f"Number of viable sets (Top 1% Likelihood): {len(viable_samples)}")

# Autocorrelation check (with quiet=True to prevent AutocorrError crash)
flat_samples = sampler.get_chain(flat=True)
tau = sampler.get_autocorr_time(quiet=True)
avg_tau = np.mean(tau)

print(f"Autocorrelation time: {tau}")
print(f"Independent samples: {len(flat_samples) / avg_tau:.0f}")

In [ ]:
def plot_lowres_spectrum_with_band(mcmc: ConstrainedMCMCHelper, viable_samples: np.ndarray, n_subsamples: int = 500) -> None:
    best_params = np.median(viable_samples, axis=0)
    wavelengths = mcmc.ObservedSpectrum.Wavelengths
    
    # Subsample the viable sets to save computation time
    indices = np.random.choice(len(viable_samples), size=min(n_subsamples, len(viable_samples)), replace=False)
    subsampled_sets = viable_samples[indices]

    all_total_fluxes = []

    # Calculate spectra for the band
    for params in subsampled_sets:
        current_total = np.zeros(len(wavelengths))
        shared_f = params[-2] * u.dex
        shared_l = params[-1] * u.dex
        
        for i in range(mcmc.number_of_components):
            w = params[i*2]
            t = params[i*2 + 1] * u.K
            comp = get_interpolated_phoenix_spectrum(t, shared_f, shared_l, star_name="fake", spec_grid=mcmc.spec_grid)
            current_total += (comp.Fluxes.value * w)
        
        all_total_fluxes.append(current_total)

    all_total_fluxes = np.array(all_total_fluxes)
    
    # Calculate the bounds of the band
    flux_min = np.min(all_total_fluxes, axis=0)
    flux_max = np.max(all_total_fluxes, axis=0)

    # --- Plotting ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True, gridspec_kw={'height_ratios': [3, 1]}, dpi=150)

    # 1. Plot the Shaded Band
    ax1.fill_between(wavelengths, flux_min, flux_max, color="orange", alpha=0.3, label="1$\sigma$ viable region")

    # 2. Plot the Ground Truth
    ax1.plot(wavelengths, mcmc.ObservedSpectrum.Fluxes, label="Original simulated spectrum", color="black", alpha=0.8, linewidth=1)

    # 3. Plot the Best Fit (Median)
    total_best_flux = np.zeros(len(wavelengths))
    best_f, best_l = best_params[-2] * u.dex, best_params[-1] * u.dex
    
    for i in range(mcmc.number_of_components):
        w, t = best_params[i*2], best_params[i*2 + 1] * u.K
        comp = get_interpolated_phoenix_spectrum(t, best_f, best_l, star_name="fake", spec_grid=mcmc.spec_grid)
        total_best_flux += (comp.Fluxes.value * w)
        
    # ax1.plot(wavelengths, total_best_flux, color="darkorange", label="Median Best Fit", linewidth=1)

    # Residuals
    res = (total_best_flux - mcmc.ObservedSpectrum.Fluxes.value) / mcmc.ObservedSpectrum.Fluxes.value
    ax2.plot(wavelengths, res, color="blue", linewidth=1)
    ax2.axhline(0, color="black", linestyle="--", alpha=0.5)

    # Formatting (Using your existing styles)
    ax1.set_ylabel("Normalised Intensity", size=14)
    ax2.set_ylabel("Residual", size=14)
    ax2.set_xlabel("Wavelength [μm]", size=14)
    ax1.legend(loc="upper right")
    
    plt.tight_layout()
    plt.show()

plot_lowres_spectrum_with_band(mcmc, viable_samples)

In [ ]:
import matplotlib.gridspec as gridspec

from scipy.spatial.distance import pdist, squareform

# Find the two most distant points in the 1-sigma region
distances = squareform(pdist(viable_samples))
i, j = np.unravel_index(np.argmax(distances), distances.shape)

params_1 = viable_samples[i]
params_2 = viable_samples[j]
print(params_1)
print(params_2)

plt.rcParams.update({'font.size': 18})
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral"],
    "mathtext.fontset": "stix",
})

def plot_parameter_comparison(mcmc, p1, p2):
    waves = mcmc.ObservedSpectrum.Wavelengths
    fig = plt.figure(figsize=(16, 12))
    gs = gridspec.GridSpec(3, 2, height_ratios=[1, 0.8, 0.25], hspace=0.3)

    ax_tl = fig.add_subplot(gs[0, 0])
    ax_tr = fig.add_subplot(gs[0, 1], sharey=ax_tl)
    ax_bot = fig.add_subplot(gs[1, :])
    ax_res = fig.add_subplot(gs[2, :], sharex=ax_bot)

    def get_full_model(params):
        comps = []
        total = np.zeros(len(waves))
        sf, sl = params[-2] * u.dex, params[-1] * u.dex
        # print("f,l")
        print(sf, sl)
        for k in range(mcmc.number_of_components):
            w, t = params[k*2], params[k*2 + 1] * u.K
            c = get_interpolated_phoenix_spectrum(t, sf, sl, star_name="fake", spec_grid=mcmc.spec_grid).Fluxes.value * w
            comps.append(c)
            total += c
        return comps, total, sf, sl

    # Generate data
    comps1, total1, feh1, logg1 = get_full_model(p1)
    comps2, total2, feh2, logg2 = get_full_model(p2)

    # Top Left: Set 1
    ax_tl.plot(waves, total1, 'k-', lw=2, label="Total")
    for idx, c in enumerate(comps1):
        f, t = p1[idx*2], p1[idx*2 + 1] * u.K
        ax_tl.plot(waves, c, '--', label=f"Component {idx+1} | f={f.round(2)}, T={t.round(2)}")
    ax_tl.set_title(f"Parameter Set 1")
    ax_tl.legend(fontsize=12)
    ax_tl.set_ylabel("Flux [Jy]")
    ax_tl.set_xlabel("Wavelength [μm]")

    # Top Right: Set 2
    ax_tr.plot(waves, total2, 'k-', lw=2, label="Total")
    for idx, c in enumerate(comps2):
        f, t = p2[idx*2], p2[idx*2 + 1] * u.K
        ax_tr.plot(waves, c, '--', label=f"Component {idx+1} | f={f.round(2)}, T={t.round(2)}")
    ax_tr.set_title(f"Parameter Set 2")
    ax_tr.legend(fontsize=12)
    ax_tr.set_xlabel("")

    # Bottom: Comparison
    ax_bot.plot(waves, total1, color="firebrick", linestyle="-", label="Total - set 1", alpha=1)
    ax_bot.plot(waves, total2 - 0.1, color="dodgerblue", linestyle="-",label="Total - set 2. Shifted down by 0.1 Jy for clarity.", alpha=1)
    ax_bot.set_ylabel("Flux [Jy]")
    ax_bot.legend()
    ax_bot.set_xlabel("")

    # Residual (A - B)
    diff = (total1 - total2)
    ax_res.plot(waves, diff, color="purple", lw=1)
    ax_res.axhline(0, color='black', linestyle=':', alpha=0.5)
    ax_res.set_ylabel("Difference [Jy]")
    ax_res.set_xlabel("Wavelength [μm]")

    for ax in [ax_tl, ax_tr, ax_bot, ax_res]:
        ax.grid(True, alpha=0.3)
        
    plt.tight_layout()
    # plt.savefig("multiple_solutions.svg", format="svg", bbox_inches='tight')
    plt.show()

plot_parameter_comparison(mcmc, params_1, params_2)

near the tail end, much similar -> more degenerate, future work could explore constraining the minimisation to certain wavelength bands

two solutions that are within 10% of the max log likelihood, but far apart in param space

when multiple components are involved, a much longer MCMC run (more steps) are needed, increasing the run time by 1 order of magnitude or so

both solutions had similar feh, logg (within 0.1 dex of each other, and within 0.15 dex of the maximum likelihood solution)
maybe should clarify that best fit means maximum likelihood, but we take the median + std dev to show the degenerate region best

old code below

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# number_of_fake_stars = 10
# number_of_components = 1

# teffs = np.linspace(min(spec_grid.T_effs), max(spec_grid.T_effs), number_of_fake_stars)

# star_list = []
# for i in range(number_of_fake_stars):
#     component_list = []
#     for j in range(number_of_components):
#         t = teffs[i]
#         w = 1
#         component_list.append(spectral_component(t, true_feh, true_logg, w))
#     star_list.append(component_list)

# def get_mcmc_results(true_comps):
#     n = len(true_comps)
#     _, _, obs = get_simulated_spectra(spec_grid, true_comps)
#     r = get_chi_r(n, NUMBER_OF_PARAMETERS, obs, spec_grid)

#     # this should definitely be done by the spectral grid class
#     parameter_bounds = [
#         (.0, 2.),
#         (np.min(spec_grid.T_effs.value), np.max(spec_grid.T_effs.value)),
#         (np.min(spec_grid.FeHs.value), np.max(spec_grid.FeHs.value)),
#         (np.min(spec_grid.Log_gs.value), np.max(spec_grid.Log_gs.value)),
#     ]

#     mcmc = ConstrainedMCMCHelper(
#         parameter_bounds=parameter_bounds,
#         number_of_components=n,
#         observed_spectrum=obs,
#         spec_grid=spec_grid,
#         n_steps=1000,
#         n_walkers=int(2.5 * n * NUMBER_OF_PARAMETERS)
#     )

#     sampler, samples = mcmc.run(r)
#     q = np.percentile(samples, [16, 50, 84], axis=0)

#     plot_corner(mcmc, sampler, samples, true_comps)
#     return q[1], q[1] - q[0], q[2] - q[1]

# res = [get_mcmc_results(s) for s in star_list]
# all_medians, all_minuses, all_pluses = zip(*res)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, ax1 = plt.subplots(1, 1, figsize=(14, 8))

for i, star in enumerate(star_list):
    true_t = star[0].T_eff.value
    t_err = [[all_minuses[i][1]], [all_pluses[i][1]]]
    
    ax1.errorbar(true_t, all_medians[i][1] - true_t, yerr=t_err, 
                 fmt='o', capsize=3, mfc='white', mec='black', ecolor='black')

# Enable minor ticks
ax1.minorticks_on()

# Adjusted grid visibility: major is now lighter/thinner than minor
ax1.grid(which='major', linestyle='-', linewidth=2, color='gray', alpha=0.5)
ax1.grid(which='minor', linestyle='--', linewidth=0.8, color='dimgray', alpha=0.6)

# Minor tick labels
ax1.xaxis.set_minor_formatter(ticker.FormatStrFormatter('%d'))
ax1.yaxis.set_minor_formatter(ticker.FormatStrFormatter('%d'))

ax1.tick_params(axis='both', which='major', labelsize=12)
ax1.tick_params(axis='both', which='minor', labelsize=8, colors='dimgrey')

# Zero line and labels
ax1.axhline(0, color='red', linestyle='-', linewidth=1, alpha=0.9)
ax1.set_ylabel(r"$T_{\mathrm{found}} - T_{\mathrm{true}}$ [K]", fontsize=14)
ax1.set_xlabel(r"$T_{\mathrm{true}}$ [K]", fontsize=14)

plt.tight_layout()
plt.show()